### Efficient TSP formulations

The TSP admits DFJ (Dantzig-Fulkerson-Johnson) subtour elimination constraints, which grow exponentially in number and can be generated only when needed. Here we use a solver-independent integer cutting-plane loop, not an in-tree branch-and-cut callback. Exploiting symmetric distances, we use one binary variable per undirected edge and require degree 2 at every city.

$$
\begin{aligned}
\text{Minimize} \quad & \sum_{i \in N} \sum_{\substack{j \in N \\ j > i}} c_{ij} x_{ij} \\
\text{subject to} \quad
& \sum_{j \in N,\; j \ne i} x_{ij} = 2 && \forall i \in N \\
& \sum_{i, j \in S} x_{ij} \leq |S| - 1 && \forall S \subset N,\; 2 \leq |S| \leq n - 1 \\
& x_{ij} = x_{ji} && \forall i < j,\; i,j \in N \\
& x_{ij} \in \{0,1\} && \forall i,j \in N,\; i \ne j
\end{aligned}
$$


In [ ]:
%pip install -q amplpy numpy matplotlib pandas networkx folium
from amplpy import AMPL, ampl_notebook
import numpy as np

# HiGHS is the default. Gurobi requires an AMPL-compatible license.
SOLVER = "highs"  # or "gurobi"
LICENSE_UUID = "default"  # Colab Community Edition; use your UUID locally
runtime = ampl_notebook(modules=[SOLVER], license_uuid=LICENSE_UUID)

def new_ampl():
    return AMPL()

def solve_checked(model):
    model.solve(solver=SOLVER)
    if model.solve_result != "solved":
        raise RuntimeError(f"No proven optimal solution: {model.solve_result}. "
                           "Inspect the solver log before extracting values.")

def values(model, name):
    # Numeric dictionaries keep plotting independent of the solver API.
    return model.var[name].get_values().to_dict()

import sys
import math
import random
from itertools import combinations
import matplotlib.pyplot as plt


In [ ]:
# Cities coordinates
cities = [
    ("Pachuca (Hidalgo)", 20.1160, -98.7330),
    ("Chetumal (Quintana Roo)", 18.5031, -88.3046),
    ("Tuxtla Gutiérrez (Chiapas)", 16.7538, -93.1131),
    ("Ciudad Victoria (Tamaulipas)", 23.7369, -99.1411),
    ("Colima (Colima)", 19.2452, -103.7249),
    ("Cuernavaca (Morelos)", 18.9242, -99.2216),
    ("Culiacán (Sinaloa)", 24.7922, -107.3940),
    ("Guadalajara (Jalisco)", 20.6597, -103.3496),
    ("Hermosillo (Sonora)", 29.0729, -110.9559),
    ("La Paz (Baja California Sur)", 24.1426, -110.3128),
    ("Mérida (Yucatán)", 20.9670, -89.6237),
    ("Mexicali (Baja California)", 32.6245, -115.4523),
    ("Monterrey (Nuevo León)", 25.6760, -100.3090),
    ("Morelia (Michoacán)", 19.7059, -101.1949),
    ("Oaxaca (Oaxaca)", 17.0732, -96.7266),
    ("Puebla (Puebla)", 19.0413, -98.2062),
    ("Querétaro (Querétaro)", 20.5888, -100.3899),
    ("San Luis Potosí (San Luis Potosí)", 22.1565, -100.9855),
    ("Tepic (Nayarit)", 21.5169, -104.8811),
    ("Tlaxcala (Tlaxcala)", 19.3191, -98.2375),
    ("Iguala (Guerrero)", 18.35, -99.5333),
    ("Xalapa (Veracruz)", 19.5438, -96.9102),
    ("Zacatecas (Zacatecas)", 22.7709, -102.5833),
    ("Villahermosa (Tabasco)", 17.9895, -92.9285),
    ("Saltillo (Coahuila)", 25.4380, -100.9737),
    ("Durango (Durango)", 24.0277, -104.6532),
    ("Guanajuato (Guanajuato)", 21.0181, -101.2574),
    ("Chihuahua (Chihuahua)", 28.6353, -106.0889),
    ("Aguascalientes (Aguascalientes)", 21.8818, -102.2950),
    ("Tlaxcala (Tlaxcala)", 19.3191, -98.2375),
    ("San Francisco de Campeche (Campeche)", 19.8454, -90.5237),
    ("Ciudad de México (Ciudad de México)", 19.4326, -99.1332),
    ("Toluca (Estado de México)", 19.2826, -99.6557)
]


n = len(cities)  # Number of cities

The first helper identifies all connected components of an optimal degree-model solution. Each proper component gives a valid DFJ cut. The second helper orders the final Hamiltonian cycle for plotting, rather than displaying an unordered connected component.

Cuts are inserted through AMPL, so the separation loop works with either HiGHS or Gurobi.

Finally, the last function returns a distance measure that is more representative than the Euclidean distance.

### Solver-independent subtour separation

The original Gurobi lazy-constraint callback is replaced by an integer cutting-plane loop. This is **not an in-tree branch-and-cut callback**: solve the degree model to optimality, identify disconnected cycles, add subtour elimination constraints, and solve again. Each solve uses the selected solver’s MIP algorithm. Once an optimal candidate is a Hamiltonian cycle, its objective proves TSP optimality because every added cut is valid for all tours.


In [ ]:
def connected_components(edges, node_count):
    adjacency = {i: set() for i in range(node_count)}
    for (i,j), value in edges.items():
        if value > 0.5:
            adjacency[i].add(j)
            adjacency[j].add(i)
    remaining = set(adjacency)
    components = []
    while remaining:
        stack = [min(remaining)]
        component = set()
        while stack:
            node = stack.pop()
            if node not in component:
                component.add(node)
                stack.extend(adjacency[node] - component)
        components.append(sorted(component))
        remaining -= component
    return components

def ordered_tour(edges, node_count):
    adjacency = {i: [] for i in range(node_count)}
    for (i,j), value in edges.items():
        if value > 0.5:
            adjacency[i].append(j)
            adjacency[j].append(i)
    route = [0]
    previous, current = None, 0
    while len(route) < node_count:
        following = next(j for j in adjacency[current] if j != previous)
        if following in route:
            raise RuntimeError("A subtour remains in the candidate solution")
        route.append(following)
        previous, current = current, following
    if 0 not in adjacency[current]:
        raise RuntimeError("Tour does not return to node 0")
    return route

# Function to compute the distance between two geographic coordinates using the Haversine formula
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth's radius in kilometers
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    distance = round(R * c)
    return distance

# Compute distances between cities using the Haversine formula
dist = {
    (i, j): haversine(cities[i][1], cities[i][2], cities[j][1], cities[j][2])
    for i in range(n) for j in range(i)
}

In [ ]:
# Solver-independent integer cutting-plane loop (NOT an in-tree callback).
# Each solve uses the selected MIP solver's branch-and-bound algorithm.
m = new_ampl()
m.eval(r"""
set N;
set E within N cross N;
param c {E} >= 0;
var e {E} binary;
minimize Total_Cost: sum {(i,j) in E} c[i,j]*e[i,j];
subject to Degree {v in N}: sum {(i,j) in E: i=v or j=v} e[i,j] = 2;
""")
m.set["N"] = list(range(n))
m.set["E"] = list(dist)
m.param["c"] = dist
cut_count = 0
iteration = 0
while True:
    iteration += 1
    solve_checked(m)  # Optimal relaxation is required for the final proof.
    vals = values(m, "e")
    components = connected_components(vals, n)
    print(f"Iteration {iteration}: {len(components)} component(s)")
    if len(components) == 1:
        break
    for component in components:
        subset = set(component)
        internal = [(i,j) for i,j in dist if i in subset and j in subset]
        lhs = " + ".join(f"e[{i},{j}]" for i,j in internal)
        m.eval(f"subject to SEC_{cut_count}: {lhs} <= {len(component)-1};")
        cut_count += 1
tour = ordered_tour(vals, n)
assert sorted(tour) == list(range(n))
print("Optimal tour:", " -> ".join(cities[i][0] for i in tour))
print("Total distance:", m.obj["Total_Cost"].value())
print("Subtour elimination cuts:", cut_count)


In [ ]:
import folium

# Map in Mexico
mexico_map = folium.Map(location=[23.6345, -102.5528], zoom_start=5)

# Marker at each capital city
for city, lat, lon in cities:
    folium.Marker([lat, lon], tooltip=city).add_to(mexico_map)

# Show mapa
mexico_map

In [ ]:
# Map in Mexico
mexico_map = folium.Map(location=[23.6345, -102.5528], zoom_start=5)

# Detect tour
optimal_tour_coords = [(cities[i][1], cities[i][2]) for i in tour]
optimal_tour_names = [cities[i][0] for i in tour]

for i in range(n):
    folium.Marker(optimal_tour_coords[i], tooltip=optimal_tour_names[i]).add_to(mexico_map)

# Draw lines
folium.PolyLine(optimal_tour_coords + [optimal_tour_coords[0]], color="blue").add_to(mexico_map)

# Show map
mexico_map